In [1]:
from typing import Any, Callable, Sequence, Tuple, Dict
import time
import jax
import jax.numpy as jnp
from jax import random
import optax
import os
from flax import linen as nn
from flax.training import train_state, checkpoints
from tqdm import tqdm
import numpy as np
from PIL import Image
import tensorflow as tf
from google.colab import drive # works only for colab
drive.mount('/content/gdrive/',)
%cd /content/gdrive/MyDrive/Facultad/tesis

Mounted at /content/gdrive/
/content/gdrive/MyDrive/Facultad/tesis


In [2]:
# -----------------------------
# Configuration / Hyperparams
# -----------------------------
NUM_CLASSES = 4
TASK_NAMES = ["denoise", "deblur", "derain", "dehaze", "enhance"]
TASK_DICT = {
    "denoise": 0,
    "deblur": 1,
    "derain": 2,
    "dehaze": 3,
    "enhance": 4,
}
PATCH_SIZE = 225
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
NUM_EPOCHS = 20
SEED = 42
LOG_EVERY = 10
CKPT_PATH = "./ckpts/classifier"
DATA_PATH = "./Datasets/Classifier"

In [3]:
# -----------------------------
# Utilities
# -----------------------------

def preprocess_pil(img: Image.Image, image_size: int = PATCH_SIZE) -> np.ndarray:
    """Convert PIL image to float32 numpy array shaped (H,W,3) normalized to [0,1]."""
    img = img.convert("RGB")
    img = img.resize((image_size, image_size), resample=Image.BILINEAR)
    arr = np.array(img).astype(np.float32) / 255.0
    # HWC format is standard for Flax/JAX
    return arr


def load_image(filepath):
    """Load and preprocess image."""
    img = Image.open(filepath).convert("RGB")
    img = np.asarray(img, np.float32) / 255.0
    return img


def make_shape_even(image):
    """Pad the image to have even shapes."""
    height, width = image.shape[0], image.shape[1]
    padh = 1 if height % 2 != 0 else 0
    padw = 1 if width % 2 != 0 else 0
    image = jnp.pad(image, [(0, padh), (0, padw), (0, 0)], mode="reflect")
    return image


def mod_padding_symmetric(image, factor=64):
    """Padding the image to be divided by factor."""
    height, width = image.shape[0], image.shape[1]
    height_pad, width_pad = (
        ((height + factor) // factor) * factor,
        ((width + factor) // factor) * factor,
    )
    padh = height_pad - height if height % factor != 0 else 0
    padw = width_pad - width if width % factor != 0 else 0
    image = jnp.pad(
        image, [(padh // 2, padh // 2), (padw // 2, padw // 2), (0, 0)], mode="reflect"
    )
    return image


def random_crop(image, crop_size):
    """Random crop for data augmentation."""
    h, w = image.shape[0], image.shape[1]

    if h > crop_size and w > crop_size:
        top = np.random.randint(0, h - crop_size)
        left = np.random.randint(0, w - crop_size)

        image = image[top : top + crop_size, left : left + crop_size, :]

    return image


def random_flip(image):
    """Random horizontal and vertical flip."""
    if np.random.rand() > 0.5:
        image = np.fliplr(image)

    if np.random.rand() > 0.5:
        image = np.flipud(image)

    return image


def random_rotation(image):
    """Random 90-degree rotation."""
    k = np.random.randint(0, 4)
    image = np.rot90(image, k=k)
    return image

In [4]:
# -----------------------------
# Model: small DW-Conv backbone
# -----------------------------
class DepthwiseConv(nn.Module):
    kernel_size: Tuple[int, int]
    strides: Tuple[int, int] = (1, 1)
    padding: str = "SAME"

    @nn.compact
    def __call__(self, x):
        in_ch = x.shape[-1]
        # Flax conv expects (N, H, W, C) by default
        # We'll use `feature_group_count=in_ch` for depthwise
        x = nn.Conv(
            features=in_ch,
            kernel_size=self.kernel_size,
            strides=self.strides,
            padding=self.padding,
            feature_group_count=in_ch,
            use_bias=False,
        )(x)
        return x


class DwSepBlock(nn.Module):
    out_ch: int
    stride: int = 1

    @nn.compact
    def __call__(self, x, train: bool = True):
        # Depthwise
        x = DepthwiseConv(kernel_size=(3, 3), strides=(self.stride, self.stride))(x)
        x = nn.BatchNorm(use_running_average=not train)(x)
        x = nn.relu(x)
        # Pointwise
        x = nn.Conv(features=self.out_ch, kernel_size=(1, 1), use_bias=False)(x)
        x = nn.BatchNorm(use_running_average=not train)(x)
        x = nn.relu(x)
        return x


class SmallBackbone(nn.Module):
    """Small lightweight backbone producing a global-pooled embedding.

    Input shape: (N, H, W, C) with C=3
    Output: (N, embedding_dim)
    """

    embedding_dim: int = 576

    @nn.compact
    def __call__(self, x, train: bool = True):
        # x: NHWC
        assert x.ndim == 4
        # initial conv
        x = nn.Conv(
            features=32,
            kernel_size=(3, 3),
            strides=(2, 2),
            padding="SAME",
            use_bias=False,
        )(x)
        x = nn.BatchNorm(use_running_average=not train)(x)
        x = nn.relu(x)

        # a few depthwise separable blocks
        x = DwSepBlock(out_ch=64, stride=1)(x, train=train)
        x = DwSepBlock(out_ch=96, stride=2)(x, train=train)
        x = DwSepBlock(out_ch=160, stride=2)(x, train=train)
        x = DwSepBlock(out_ch=self.embedding_dim, stride=2)(x, train=train)

        # global avg pool
        # x shape: (N, H, W, C)
        x = x.mean(axis=(1, 2))  # (N, C)
        return x


class RouterHead(nn.Module):
    num_classes: int = NUM_CLASSES

    @nn.compact
    def __call__(self, x):
        x = nn.Dense(256)(x)
        x = nn.relu(x)
        x = nn.Dense(128)(x)
        x = nn.relu(x)
        x = nn.Dense(self.num_classes)(x)
        return x


class RouterModel(nn.Module):
    embedding_dim: int = 576
    num_classes: int = NUM_CLASSES

    def setup(self):
        self.backbone = SmallBackbone(embedding_dim=self.embedding_dim)
        self.head = RouterHead(num_classes=self.num_classes)

    def __call__(self, x, train: bool = True):
        feats = self.backbone(x, train=train)
        logits = self.head(feats)
        return logits

In [5]:
# -----------------------------
# Train state / Init / Trainer
# -----------------------------


class TrainState(train_state.TrainState):
    batch_stats: Any = None  # for BatchNorm

def create_train_state(rng, learning_rate=LEARNING_RATE):
    model = RouterModel()
    input_shape = (1, PATCH_SIZE, PATCH_SIZE, 3)
    variables = model.init(rng, jnp.ones(input_shape, jnp.float32), train=True)
    params = variables["params"]
    batch_stats = variables.get("batch_stats")

    tx = optax.adamw(learning_rate=learning_rate, weight_decay=WEIGHT_DECAY)
    state = TrainState.create(
        apply_fn=model.apply, params=params, tx=tx, batch_stats=batch_stats
    )
    return state


# -----------------------------
# Loss / metrics
# -----------------------------


def cross_entropy_loss(logits, labels):
    onehot = jax.nn.one_hot(labels, logits.shape[-1])
    loss = optax.softmax_cross_entropy(logits=logits, labels=onehot)
    return loss.mean()


@jax.jit
def compute_metrics(logits, labels):
    loss = cross_entropy_loss(logits, labels)
    acc = jnp.mean(jnp.argmax(logits, -1) == labels)
    return {"loss": loss, "accuracy": acc}


# -----------------------------
# Training / Eval step
# -----------------------------

@jax.jit
def train_step(state: TrainState, batch: Tuple[jnp.ndarray, jnp.ndarray], rng):
    imgs, labels = batch  # imgs: NHWC float32, labels: (N,)

    def loss_fn(params):
        logits, new_model_state = state.apply_fn(
            {"params": params, "batch_stats": state.batch_stats},
            imgs,
            train=True,
            mutable=["batch_stats"],
        )
        loss = cross_entropy_loss(logits, labels)
        return loss, (logits, new_model_state)

    grad_fn = jax.value_and_grad(loss_fn, has_aux=True)
    (loss, (logits, new_model_state)), grads = grad_fn(state.params)
    state = state.apply_gradients(
        grads=grads, batch_stats=new_model_state["batch_stats"]
    )
    metrics = compute_metrics(logits, labels)
    return state, metrics

def train_epoch(state: TrainState, train_dataset, epoch):
    """Train for one epoch."""
    batch_metrics = []
    print(f"Starting training epoch {epoch + 1}")
    rng, init_rng = jax.random.split(jax.random.PRNGKey(SEED))
    for step, (images, labels) in enumerate(train_dataset):
        rng, step_rng = jax.random.split(rng)
        images = jnp.array(images)
        labels = jnp.array(labels)
        if images.shape[0] != labels.shape[0]:
            print(f"Skipping step {step} due to batch size mismatch: input {images.shape[0]}, target {labels.shape[0]}")
            continue

        rng, step_rng = jax.random.split(rng)
        state, metrics = train_step(state, (images, labels), step_rng)
        batch_metrics.append(metrics)

        if (step + 1) % LOG_EVERY == 0:
            metrics_np = jax.device_get(metrics)
            print(
                f"Epoch {epoch}, Step {step + 1}: "
                f'loss = {metrics_np["loss"]:.4f}, '
                f'accuracy = {metrics_np["accuracy"]:.2f} dB'
            )

    # Compute epoch metrics
    epoch_metrics = {
        k: np.mean([m[k] for m in batch_metrics]) for k in batch_metrics[0].keys()
    }

    return state, epoch_metrics


@jax.jit
def eval_step(state: TrainState, images: jnp.ndarray, labels: jnp.ndarray):
    variables = {"params": state.params, "batch_stats": state.batch_stats}
    logits = state.apply_fn(variables, images, train=False)
    metrics = compute_metrics(logits, labels)
    return metrics

def evaluate(state, val_dataset):
    """Evaluate on validation set."""
    batch_metrics = []

    # Use tqdm for progress bar
    pbar = tqdm(val_dataset, desc="Evaluating")
    expected_batch_size = BATCH_SIZE
    for images, labels in pbar:
        images = jnp.asarray(images, dtype=jnp.float32)
        labels = jnp.asarray(labels, dtype=jnp.int32)

        current_batch_size = images.shape[0]
        # Pad if necessary to avoid recompilation
        if current_batch_size < expected_batch_size:
            pad_amount = expected_batch_size - current_batch_size
            pad_width = [(0, pad_amount)] + [(0, 0)] * (images.ndim - 1)
            # Use edge padding for target to avoid huge errors
            images = jnp.pad(images, pad_width, mode='edge')
            labels = jnp.pad(labels, pad_width, mode='edge')

        metrics = eval_step(state, images, labels)
        metrics_np = jax.device_get(metrics)
        batch_metrics.append((metrics_np, current_batch_size))

    # Compute average metrics
    if not batch_metrics:
        return {}

    avg_metrics = {}
    total_samples = sum(count for _, count in batch_metrics)

    # Get keys from first batch
    keys = batch_metrics[0][0].keys()

    for k in keys:
        # Weighted average
        weighted_sum = sum(m[k] * count for m, count in batch_metrics)
        avg_metrics[k] = weighted_sum / total_samples

    return avg_metrics


# -----------------------------
# Dataset helpers
# -----------------------------

def read_lines_from_file(basepath, task, is_training):
    file = "train.txt" if is_training else "test.txt"
    print(f"reading {file} data for task:{task}")
    filepath = os.path.join(basepath, task , file)
    with open(filepath, "r", encoding="utf-8") as f:
        paths = [line.strip() for line in f if line.strip()]

    imgs_dir = os.path.join(basepath, task, "imgs")
    for path in paths:
        p = os.path.join(imgs_dir, path)
        if os.path.exists(p):
            yield p


def create_dataset(data_dir, batch_size, is_training=True):
    """Create TensorFlow dataset for training/validation."""
    print(
        f"Creating {'training' if is_training else 'validation'} dataset from {data_dir}"
    )
    images = []
    labels = []

    for task in TASK_DICT:
        img_paths = read_lines_from_file(data_dir, task, is_training)
        for img_path in img_paths:
            images.append(img_path)
            labels.append(TASK_DICT[task])

    def load_and_preprocess(input_path, label):
        """Load and preprocess a single pair of images."""
        input_img = preprocess_pil(Image.open(input_path.numpy().decode()))
        # Padding images to have even shapes
        input_img = make_shape_even(input_img)
        # Padding images to be multiples of 64
        input_img = mod_padding_symmetric(input_img, factor=64)

        if is_training:
            # Data augmentation
            input_img = random_crop(input_img, PATCH_SIZE)
            input_img = random_flip(input_img)
            input_img = random_rotation(input_img)

        return (
            input_img.astype(np.float32),
            tf.cast(label, tf.int32),
        )

    dataset = tf.data.Dataset.from_tensor_slices((images, labels))

    if is_training:
        dataset = dataset.shuffle(buffer_size=1000)

    dataset = dataset.map(
        lambda x, y: tf.py_function(
            func=load_and_preprocess,
            inp=[x, y],
            Tout=[tf.float32, tf.int32],
        ),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    dataset = dataset.batch(batch_size, drop_remainder=is_training)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset, len(images)


def data_loader(
    images: np.ndarray, labels: np.ndarray, batch_size=BATCH_SIZE, shuffle=True
):
    n = len(images)
    idxs = np.arange(n)
    if shuffle:
        np.random.shuffle(idxs)
    for i in range(0, n, batch_size):
        batch_idx = idxs[i : i + batch_size]
        yield images[batch_idx], labels[batch_idx]

# -----------------------------
# Saving / Loading
# -----------------------------


def save_checkpoint(state: TrainState, step: int):
    save_dict = {
        "params": state.params,
        "batch_stats": state.batch_stats,
        "opt_state": state.opt_state,
    }
    checkpoints.save_checkpoint(
        ckpt_dir=CKPT_PATH, target=save_dict, step=step, overwrite=True
    )


def load_checkpoint(state: TrainState):
    ckpt = checkpoints.restore_checkpoint(ckpt_dir=CKPT_PATH, target=None)
    if ckpt:
        state = state.replace(
            params=ckpt["params"], batch_stats=ckpt.get("batch_stats")
        )
    return state


# -----------------------------
# Inference helper
# -----------------------------


def route_image_pil(
    state: TrainState, pil_img: Image.Image, rng=None
) -> Tuple[str, float]:
    arr = preprocess_pil(pil_img)  # HWC float32
    arr = jnp.array(arr)
    arr = arr[None, ...]  # 1HWC
    variables = {"params": state.params, "batch_stats": state.batch_stats}
    logits = state.apply_fn(variables, arr, train=False, mutable=False)
    probs = jax.nn.softmax(logits, axis=-1)
    probs = np.array(probs[0])
    idx = int(np.argmax(probs))
    return TASK_NAMES[idx], float(probs[idx])


In [6]:
# -----------------------------
# Training Loop
# -----------------------------

rng = random.PRNGKey(SEED)
state = create_train_state(rng)

train_dataset, train_size = create_dataset(
    data_dir=DATA_PATH, batch_size=BATCH_SIZE, is_training=True
)
test_dataset, test_size = create_dataset(
    data_dir=DATA_PATH, batch_size=BATCH_SIZE, is_training=False
)

best_acc = 0.0
print(f"Training for {range(NUM_EPOCHS)} epochs...")
for epoch in range(NUM_EPOCHS):
    print(f"Epoch {epoch + 1}/{NUM_EPOCHS}")
    state, train_metrics = train_epoch(state, train_dataset, epoch)

    print(
        f"Epoch {epoch} training: "
        f'loss = {train_metrics["loss"]:.4f}, '
        f'accuracy = {train_metrics["accuracy"]:.2f} dB'
    )

    # Validation
    val_metrics = evaluate(state, test_dataset.take(15))
    print(
        f"Epoch {epoch} validation: "
        f'loss = {val_metrics["loss"]:.4f}, '
        f'accuracy = {val_metrics["accuracy"]:.2f} dB'
    )

    if (epoch + 1) % 10 == 0 or val_metrics["accuracy"] > best_acc:
        ckpt_path = os.path.join(CKPT_PATH, f"checkpoint_{epoch}")
        checkpoints.save_checkpoint(
            ckpt_dir=ckpt_path,
            target={"opt": {"target": state.params}},
            step=epoch,
            overwrite=True,
        )
        print(f"Saved checkpoint to {ckpt_path}")

        if val_metrics["accuracy"] > best_acc:
            best_acc = val_metrics["accuracy"]
            best_ckpt_path = os.path.join(CKPT_PATH, "best_checkpoint")
            checkpoints.save_checkpoint(
                ckpt_dir=best_ckpt_path,
                target={"opt": {"target": state.params}},
                step=epoch,
                overwrite=True,
            )
            print(f"New best model! ACC: {best_acc:.2f} dB")

print("Training completed!")

Creating training dataset from ./Datasets/Classifier
reading train.txt data for task:denoise
reading train.txt data for task:deblur
reading train.txt data for task:derain
reading train.txt data for task:dehaze
reading train.txt data for task:enhance
Creating validation dataset from ./Datasets/Classifier
reading test.txt data for task:denoise
reading test.txt data for task:deblur
reading test.txt data for task:derain
reading test.txt data for task:dehaze
reading test.txt data for task:enhance
Training for range(0, 20) epochs...
Epoch 1/20
Starting training epoch 1
Epoch 0, Step 10: loss = 0.2330, accuracy = 0.95 dB
Epoch 0, Step 20: loss = 0.1351, accuracy = 0.97 dB
Epoch 0, Step 30: loss = 0.1174, accuracy = 0.98 dB
Epoch 0, Step 40: loss = 0.0029, accuracy = 1.00 dB
Epoch 0, Step 50: loss = 0.0023, accuracy = 1.00 dB
Epoch 0, Step 60: loss = 0.0017, accuracy = 1.00 dB
Epoch 0, Step 70: loss = 0.0033, accuracy = 1.00 dB
Epoch 0, Step 80: loss = 0.0013, accuracy = 1.00 dB
Epoch 0, Step 

Evaluating: 100%|██████████| 15/15 [01:07<00:00,  4.50s/it]


Epoch 0 validation: loss = 10.9977, accuracy = 0.00 dB
Epoch 2/20
Starting training epoch 2
Epoch 1, Step 10: loss = 0.9539, accuracy = 0.92 dB
Epoch 1, Step 20: loss = 0.1518, accuracy = 0.97 dB
Epoch 1, Step 30: loss = 0.0690, accuracy = 0.98 dB
Epoch 1, Step 40: loss = 0.0231, accuracy = 1.00 dB
Epoch 1, Step 50: loss = 0.0159, accuracy = 1.00 dB
Epoch 1, Step 60: loss = 0.0060, accuracy = 1.00 dB
Epoch 1, Step 70: loss = 0.0035, accuracy = 1.00 dB
Epoch 1, Step 80: loss = 0.0030, accuracy = 1.00 dB
Epoch 1, Step 90: loss = 0.0014, accuracy = 1.00 dB
Epoch 1, Step 100: loss = 0.0005, accuracy = 1.00 dB
Epoch 1, Step 110: loss = 0.0006, accuracy = 1.00 dB
Epoch 1, Step 120: loss = 0.0005, accuracy = 1.00 dB
Epoch 1, Step 130: loss = 1.0966, accuracy = 0.61 dB
Epoch 1, Step 140: loss = 0.3324, accuracy = 0.98 dB
Epoch 1, Step 150: loss = 0.0542, accuracy = 0.97 dB
Epoch 1, Step 160: loss = 0.1360, accuracy = 0.95 dB
Epoch 1, Step 170: loss = 0.0941, accuracy = 0.94 dB
Epoch 1, Step 18

Evaluating: 100%|██████████| 15/15 [00:10<00:00,  1.44it/s]


Epoch 1 validation: loss = 11.7185, accuracy = 0.00 dB
Epoch 3/20
Starting training epoch 3
Epoch 2, Step 10: loss = 1.2567, accuracy = 0.00 dB
Epoch 2, Step 20: loss = 0.7021, accuracy = 0.95 dB
Epoch 2, Step 30: loss = 0.0255, accuracy = 1.00 dB
Epoch 2, Step 40: loss = 0.0986, accuracy = 0.98 dB
Epoch 2, Step 50: loss = 0.0798, accuracy = 0.98 dB
Epoch 2, Step 60: loss = 0.0056, accuracy = 1.00 dB
Epoch 2, Step 70: loss = 0.0048, accuracy = 1.00 dB
Epoch 2, Step 80: loss = 0.0029, accuracy = 1.00 dB
Epoch 2, Step 90: loss = 0.0020, accuracy = 1.00 dB
Epoch 2, Step 100: loss = 0.0018, accuracy = 1.00 dB
Epoch 2, Step 110: loss = 0.0010, accuracy = 1.00 dB
Epoch 2, Step 120: loss = 0.0007, accuracy = 1.00 dB
Epoch 2, Step 130: loss = 1.2048, accuracy = 0.72 dB
Epoch 2, Step 140: loss = 0.7849, accuracy = 0.92 dB
Epoch 2, Step 150: loss = 0.0937, accuracy = 0.97 dB
Epoch 2, Step 160: loss = 0.0411, accuracy = 0.98 dB
Epoch 2, Step 170: loss = 0.0338, accuracy = 0.98 dB
Epoch 2, Step 18

...

Epoch 8 validation: loss = 2.6524, accuracy = 0.00 dB
Epoch 10/20
Starting training epoch 10
Epoch 9, Step 10: loss = 1.3572, accuracy = 0.36 dB
Epoch 9, Step 20: loss = 0.7008, accuracy = 0.95 dB
Epoch 9, Step 30: loss = 0.3710, accuracy = 0.98 dB
Epoch 9, Step 40: loss = 0.2349, accuracy = 0.97 dB
Epoch 9, Step 50: loss = 0.0255, accuracy = 1.00 dB
Epoch 9, Step 60: loss = 0.0048, accuracy = 1.00 dB
Epoch 9, Step 70: loss = 0.0025, accuracy = 1.00 dB
Epoch 9, Step 80: loss = 0.0019, accuracy = 1.00 dB
Epoch 9, Step 90: loss = 0.0018, accuracy = 1.00 dB
Epoch 9, Step 100: loss = 0.0016, accuracy = 1.00 dB
Epoch 9, Step 110: loss = 0.0010, accuracy = 1.00 dB
Epoch 9, Step 120: loss = 0.0012, accuracy = 1.00 dB
Epoch 9, Step 130: loss = 1.3721, accuracy = 0.66 dB
Epoch 9, Step 140: loss = 0.6627, accuracy = 0.86 dB
Epoch 9, Step 150: loss = 0.7221, accuracy = 0.69 dB
Epoch 9, Step 160: loss = 0.6713, accuracy = 0.66 dB
Epoch 9, Step 170: loss = 0.2432, accuracy = 0.88 dB
Epoch 9, Step 1

Evaluating: 100%|██████████| 15/15 [00:10<00:00,  1.43it/s]


Epoch 9 validation: loss = 1.5590, accuracy = 0.25 dB


ERROR:absl:[process=0][thread=array_type_handler][operation_id=1] _SignalingThread.run() raised an exception: Checkpoint path should be absolute. Got ckpts/classifier/checkpoint_9/checkpoint_9.orbax-checkpoint-tmp
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/orbax/checkpoint/_src/futures/future.py", line 312, in run
    super().run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/orbax/checkpoint/_src/futures/future.py", line 257, in _target_setting_result
    self._result = target()
                   ^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/orbax/checkpoint/_src/futures/future.py", line 391, in <lambda>
    target=lambda: asyncio_utils.run_sync(coro),
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/orbax/checkpoint/_src/asyncio_utils.py", line 36, in run_sync
    return asyncio

ValueError: Checkpoint path should be absolute. Got ckpts/classifier/checkpoint_9/checkpoint_9.orbax-checkpoint-tmp